# DS 340W - Reproducing HIWL (Ma et al., 2023, MNRAS)

Run on Kaggle with GPU (T4 x2) and the Galaxy Zoo dataset attached as input.

## Step 1 - Setup: clone this repo

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
!git clone https://github.com/asamad0687/HIWL-DS340W.git

In [ ]:
%cd HIWL-DS340W/scheme

## Step 2 - Data: check the Galaxy Zoo input files

In [ ]:
!ls /kaggle/input/notebooks/kingjuno/galaxy-zoo

## Step 3 - Clean sample selection (`select cleandata.py`)
Only the three hardcoded Windows paths are swapped for Kaggle paths at runtime. No logic is changed.

In [ ]:
with open("../select cleandata.py", "r") as f:
    content = f.read()

content = content.replace(
    r"F:\dataSet\galaxy-zoo-the-galaxy-challenge\images_training_rev1",
    "/kaggle/input/notebooks/kingjuno/galaxy-zoo/images_training_rev1"
)
content = content.replace(
    r"F:\dataSet\galaxy-zoo-the-galaxy-challenge\training_solutions_rev1.csv",
    "/kaggle/input/notebooks/kingjuno/galaxy-zoo/training_solutions_rev1.csv"
)
content = content.replace(
    r"F:\dataSet\clean gzdata5",
    "/kaggle/working/clean_gzdata5"
)

with open("../select cleandata.py", "w") as f:
    f.write(content)

print("Paths updated.")

In [ ]:
!cat "../select cleandata.py"

In [ ]:
!python "../select cleandata.py"

### Check the class counts against the paper's Table 2
Expected: rou 8434, bet 8069, cig 578, edg 3903, spi 7806 (28,790 total)

In [ ]:
!ls /kaggle/working/clean_gzdata5

In [ ]:
!ls /kaggle/working/clean_gzdata5/rou | head -5
!ls /kaggle/working/clean_gzdata5/rou | wc -l

In [ ]:
!ls /kaggle/working/clean_gzdata5/bet | wc -l   # expect 8069 (in-between smooth)
!ls /kaggle/working/clean_gzdata5/cig | wc -l   # expect 578 (cigar-shaped)
!ls /kaggle/working/clean_gzdata5/edg | wc -l   # expect 3903 (edge-on)
!ls /kaggle/working/clean_gzdata5/spi | wc -l   # expect 7806 (spiral)

## Step 4 - Training utilities
`pytorchtools.py` (EarlyStopping) was missing from the authors' upload, so it's recreated here.

In [ ]:
!cat utils.py

In [ ]:
early_stopping_code = '''
import numpy as np
import torch

class EarlyStopping:
    """Stops training if validation accuracy doesn't improve after a given patience."""
    def __init__(self, patience=25, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.delta = delta

    def __call__(self, val_loss, val_acc, model):
        score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.val_acc_max = val_acc
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.val_acc_max = val_acc
            self.counter = 0
'''

with open("pytorchtools.py", "w") as f:
    f.write(early_stopping_code)

print("pytorchtools.py created.")

## Step 5 - Train HIWL (EfficientNet-B1)

In [ ]:
%cd /kaggle/working/HIWL-DS340W/scheme
!python train_efficientnet.py --data-path5 /kaggle/working/clean_gzdata5 --weights4 "" --weights2 "" --os linux --device cuda:0 --num-model B1

## Step 6 - Save the train / validation / test split (8:1:1) for later use

In [ ]:
from utils import read_split_data

train_images_path, train_images_label, val_images_path, val_images_label, test_images_path, test_images_label = read_split_data("/kaggle/working/clean_gzdata5")

import pandas as pd

pd.DataFrame({"path": train_images_path, "label": train_images_label}).to_csv("/kaggle/working/train_set.csv", index=False)
pd.DataFrame({"path": val_images_path, "label": val_images_label}).to_csv("/kaggle/working/test_set.csv", index=False)      # paper's "val" = our Test Set
pd.DataFrame({"path": test_images_path, "label": test_images_label}).to_csv("/kaggle/working/validation_unseen_set.csv", index=False)  # paper's "test" = our Validation (Unseen) Set

print("Files saved.")

## Step 7 - Evaluate: compare with the paper's Table 4
Prints overall accuracy, per-class recall / precision / F1 next to the paper's numbers, and the confusion matrix. Saves `results/metrics_B1.csv`.

In [ ]:
%cd /kaggle/working/HIWL-DS340W/scheme
!python evaluate_metrics.py --data-path5 /kaggle/working/clean_gzdata5 --num-model B1

## Step 8 - Novel approach (coming next)
The confidence scoring and prediction stability modules will go in `novel/` and use the saved model weights and predictions from Step 5.